In [58]:
import pandas as pd
import os

BASE = os.path.dirname(os.getcwd())
DIR = os.path.join(BASE, "data-round-1")

# Q1: Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

In [59]:
orders = pd.read_csv(os.path.join(DIR, "orders.csv"))
orders.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign


In [60]:
# Parse date
orders['order_date'] = pd.to_datetime(orders["order_date"])

# Sort by customer và date
orders = orders.sort_values(["customer_id", "order_date"])

# Tính gap giữa các đơn liên tiếp của cùng customer
orders["prev_date"] = orders.groupby("customer_id")["order_date"].shift(1)
orders["gap_days"] = (orders["order_date"] - orders["prev_date"]).dt.days

# Chỉ lấy customers có >1 đơn (tức là có gap_days không null)
gaps = orders.dropna(subset=["gap_days"])

# Lọc thêm: chỉ lấy customers thực sự có >1 order
multi_order_customers = orders.groupby("customer_id").filter(lambda x: len(x) > 1)
gaps = multi_order_customers.dropna(subset=["gap_days"])

median_gap = gaps["gap_days"].median()
print(f"Median inter-order gap: {median_gap} days")

Median inter-order gap: 144.0 days


-> Q1 chọn C ~180 ngày

# Q2: Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp trung bình cao nhất, với công thức (price − cogs)/price?

In [61]:
products = pd.read_csv(os.path.join(DIR, "products.csv"))
products.head()

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406


In [62]:
products.segment.value_counts()

segment
Activewear     598
Everyday       405
Performance    347
Balanced       306
Standard       262
Premium        177
All-weather    169
Trendy         148
Name: count, dtype: int64

In [63]:
products["helper"] = (products["price"] - products["cogs"]) / products["price"]
result = products.groupby("segment")["helper"].mean().sort_values(ascending = False)
print(result)

segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: helper, dtype: float64


-> Q2 chọn A : Standard

# Q3: Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

In [64]:
returns = pd.read_csv(os.path.join(DIR, "returns.csv"))
products = pd.read_csv(os.path.join(DIR, "products.csv"))

In [65]:
returns.head()

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76


In [66]:
products.head()

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406


In [67]:
merged = pd.merge(returns, products, on = "product_id", how = "inner")
result = merged.loc[merged.category == "Streetwear"]["return_reason"].value_counts(sort = True)

In [68]:
result

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

-> Q3 chọn B : Wrong size

# Q4: Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

In [69]:
web_traffic = pd.read_csv(os.path.join(DIR, "web_traffic.csv"))
web_traffic.head()

,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct
3,2013-01-04,9973,8063,53078,0.00562,151.8,direct
4,2013-01-05,10223,7882,36790,0.00525,168.6,referral


In [70]:
result = web_traffic.groupby("traffic_source")["bounce_rate"].mean().sort_values(ascending = True)
result

traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64

-> Q4 chọn C : Email campaign

# Q5 : Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id không null) xấp xỉ là bao nhiêu?

In [71]:
order_items = pd.read_csv(os.path.join(DIR, "order_items.csv"))
order_items.head()

C:\Users\PC\AppData\Local\Temp\ipykernel_16720\1972626635.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(os.path.join(DIR, "order_items.csv"))


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2400,7,1138.22,0.0,NaN,NaN
1,2,609,7,10166.25,0.0,NaN,NaN
2,3,396,3,11220.33,0.0,NaN,NaN
3,4,635,5,10639.25,0.0,NaN,NaN
4,6,1935,1,1597.84,0.0,NaN,NaN


In [72]:
result = 100 - order_items["promo_id"].isnull().sum() * 100 / len(order_items)
result

38.663493169565214

-> Q5 chọn C : ~39%

# Q6 : Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong nhóm)

In [73]:
customers = pd.read_csv(os.path.join(DIR, "customers.csv"))
orders = pd.read_csv(os.path.join(DIR, "orders.csv"))

In [74]:
customers.head()

,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search
3,4,15201,Hai Phong,2017-11-29,Male,35-44,referral
4,5,15201,Hai Phong,2022-09-23,Male,55+,organic_search


In [75]:
orders.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign


In [76]:
merge = pd.merge(customers, orders, on = "customer_id", how = "inner")
merge.head()

,customer_id,zip_x,city,signup_date,gender,age_group,acquisition_channel,order_id,order_date,zip_y,order_status,payment_method,device_type,order_source
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media,5280,2012-07-25,15201,delivered,cod,desktop,paid_search
1,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media,184922,2014-05-31,15201,returned,credit_card,mobile,referral
2,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media,308113,2015-07-31,15201,delivered,cod,mobile,paid_search
3,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media,483190,2017-04-23,15201,delivered,cod,mobile,paid_search
4,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media,702081,2020-02-24,15201,delivered,credit_card,mobile,organic_search


In [77]:
result = merge.groupby("age_group").apply(
    lambda x: len(x) / x["customer_id"].nunique()
).sort_values(ascending = False)
result

C:\Users\PC\AppData\Local\Temp\ipykernel_16720\1478778062.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = merge.groupby("age_group").apply(


age_group
55+      7.268731
45-54    7.220264
35-44    7.206159
25-34    7.112230
18-24    7.068577
dtype: float64

-> Q6 chọn A : 55+

# Q7: Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong sales_train.csv?

In [78]:
orders    = pd.read_csv(os.path.join(DIR, "orders.csv"))
payments  = pd.read_csv(os.path.join(DIR, "payments.csv"))
geography = pd.read_csv(os.path.join(DIR, "geography.csv"))

order_payments = orders[["order_id", "zip"]].merge(
    payments[["order_id", "payment_value"]],
    on="order_id",
    how="inner"
)

order_payments_region = order_payments.merge(
    geography[["zip", "region"]].drop_duplicates("zip"),
    on="zip",
    how="inner"
)

revenue_by_region = (
    order_payments_region
    .groupby("region", dropna=False)["payment_value"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"payment_value": "total_revenue"})
)

print(revenue_by_region)
print(f"\n→ Region doanh thu cao nhất: {revenue_by_region.iloc[0]['region']} "
      f"({revenue_by_region.iloc[0]['total_revenue']:,.0f})")

    region  total_revenue
0     East   7.291151e+09
1  Central   4.719491e+09
2     West   3.670227e+09

→ Region doanh thu cao nhất: East (7,291,150,819)


-> Q7 chọn C : East

# Q8 : Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức thanh toán nào được sử dụng nhiều nhất?

In [79]:
orders = pd.read_csv(os.path.join(DIR, "orders.csv"))
orders.head()

,order_id,order_date,customer_id,zip,order_status,payment_method,device_type,order_source
0,1,2012-07-04,58578,1109,delivered,credit_card,desktop,paid_search
1,2,2012-07-04,58621,1330,returned,cod,mobile,paid_search
2,3,2012-07-04,58811,1473,delivered,credit_card,desktop,direct
3,4,2012-07-04,59453,2360,delivered,credit_card,desktop,referral
4,6,2012-07-06,57821,2886,delivered,paypal,mobile,email_campaign


In [80]:
result = orders.loc[orders.order_status == "cancelled"]["payment_method"].value_counts(sort = True)
result

payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64

-> Q8 chọn A : Credit card

# Q9: Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join với products theo product_id)?

In [81]:
returns = pd.read_csv(os.path.join(DIR, "returns.csv"))
order_items = pd.read_csv(os.path.join(DIR, "order_items.csv"))
products = pd.read_csv(os.path.join(DIR, "products.csv"))

C:\Users\PC\AppData\Local\Temp\ipykernel_16720\2887244238.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv(os.path.join(DIR, "order_items.csv"))


In [82]:
returns.head()

,return_id,order_id,product_id,return_date,return_reason,return_quantity,refund_amount
0,RET-000001,2,609,2012-07-25,late_delivery,6,52458.01
1,RET-000002,32,1862,2012-07-16,wrong_size,2,5141.37
2,RET-000003,35,2359,2012-07-16,wrong_size,1,5315.95
3,RET-000004,47,1449,2012-07-11,wrong_size,4,6493.75
4,RET-000005,47,1450,2012-07-25,wrong_size,1,1740.76


In [83]:
order_items.head()

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2400,7,1138.22,0.0,NaN,NaN
1,2,609,7,10166.25,0.0,NaN,NaN
2,3,396,3,11220.33,0.0,NaN,NaN
3,4,635,5,10639.25,0.0,NaN,NaN
4,6,1935,1,1597.84,0.0,NaN,NaN


In [84]:
products.head()

,product_id,product_name,category,segment,size,color,price,cogs
0,536,SaigonFlex UC-01,Streetwear,Everyday,S,green,11059.650000,9704.842875
1,537,SaigonFlex UC-02,Streetwear,Everyday,M,silver,9523.076013,5393.870254
2,538,SaigonFlex UC-03,Streetwear,Everyday,L,pink,15951.633158,11371.919278
3,539,SaigonFlex UC-04,Streetwear,Everyday,XL,yellow,15753.717299,8573.172954
4,540,SaigonFlex UC-05,Streetwear,Everyday,S,red,15766.334536,14063.570406


In [85]:
returns_products = pd.merge(returns, products, on = "product_id", how = "inner")
order_items_products = pd.merge(order_items, products, on = "product_id", how = "inner")

In [86]:
returns_products["size"].value_counts()

size
XL    10655
M      9820
L      9741
S      9723
Name: count, dtype: int64

In [87]:
order_items_products["size"].value_counts()

size
XL    193025
M     176428
L     173174
S     172042
Name: count, dtype: int64

In [88]:
returns_products["size"].value_counts() * 100 / order_items_products["size"].value_counts()

size
XL    5.520010
M     5.566010
L     5.624978
S     5.651527
Name: count, dtype: float64

-> Q9 chọn A : S

# Q10: Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất? 

In [89]:
payments = pd.read_csv(os.path.join(DIR, "payments.csv"))
payments.head()

,order_id,payment_method,payment_value,installments
0,1,credit_card,7967.54,3
1,2,cod,71163.75,1
2,3,credit_card,33660.99,3
3,4,credit_card,53196.25,3
4,6,paypal,1597.84,1


In [90]:
result = payments.groupby("installments").apply(
    lambda x: x.payment_value.sum() / x["order_id"].nunique()
).sort_values(ascending = False)
result

C:\Users\PC\AppData\Local\Temp\ipykernel_16720\3654937167.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = payments.groupby("installments").apply(


installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
dtype: float64

-> Q10 chọn C : 6 kỳ